# Oracle Ceiling Computation and Visualisation

This notebook computes the **oracle ceiling** — the accuracy a hypothetical perfect selector would achieve if it always picked the correct answer whenever *either* the text agent or the vision agent got it right.

**Input:** A CSV export from any W&B run that has per-agent correctness columns (e.g., the Mode 1 CoT run).

**Output:** The oracle ceiling value and a clear visualisation for the thesis.

In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use if on HPC without display
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

## 1. Load Data

Update the CSV path below to point to your exported W&B results table.
The CSV needs columns for `image_id`, `text_correct` (or `text_predicted` + `ground_truth`), and `vision_correct` (or `vision_predicted` + `ground_truth`).

In [13]:
# ── UPDATE THIS PATH ──
CSV_PATH = "wandb_exports/mode3b_structured_debate_689.csv"  # Your exported W&B CSV

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} cases")
print(f"Columns: {list(df.columns)}")
df.head(3)

Loaded 689 cases
Columns: ['image_id', 'question', 'ground_truth', 'brier_score', 'text_r1_answer', 'text_r1_correct', 'text_r1_confidence', 'vision_r1_answer', 'vision_r1_correct', 'vision_r1_confidence', 'r1_agree', 'text_r2_answer', 'text_r2_correct', 'text_r2_confidence', 'vision_r2_answer', 'vision_r2_correct', 'vision_r2_confidence', 'r2_agree', 'text_changed', 'text_change_direction', 'vision_changed', 'vision_change_direction', 'text_conf_delta', 'vision_conf_delta', 'meta_answer', 'meta_correct', 'text_r1_parse', 'text_r2_parse', 'vision_r1_parse', 'vision_r2_parse', 'text_r1_raw', 'vision_r1_raw', 'text_r2_raw', 'vision_r2_raw', 'meta_output', 'running_meta_accuracy', 'running_text_r1_accuracy', 'running_text_r2_accuracy', 'running_vision_r1_accuracy', 'running_vision_r2_accuracy']


,image_id,question,ground_truth,brier_score,text_r1_answer,text_r1_correct,text_r1_confidence,vision_r1_answer,vision_r1_correct,vision_r1_confidence,...,text_r1_raw,vision_r1_raw,text_r2_raw,vision_r2_raw,meta_output,running_meta_accuracy,running_text_r1_accuracy,running_text_r2_accuracy,running_vision_r1_accuracy,running_vision_r2_accuracy
0,13,This 36-year-old man has undergone renal trans...,A,0.094,A,correct,0.95,C,wrong,0.95,...,<unused94>thought\nThe user wants me to analyz...,<unused94>thought\nThe user wants me to analyz...,"```json\n{\n ""answer"": ""C"",\n ""confidence"": ...",<unused94>thought\nThe user wants me to analyz...,ANSWER: C\nREASONING: Both specialists agree o...,0.0000,1.0,0.0000,0.0000,1.0
1,556,A 48-year-old woman who lived in rural India p...,B,0.076,B,correct,0.98,D,wrong,0.95,...,<unused94>thought\nThe user wants me to identi...,<unused94>thought\nThe user wants me to identi...,<unused94>thought\nThe user wants me to analyz...,<unused94>thought\nThe user wants me to compar...,ANSWER: B\nREASONING: Both specialists reached...,0.5000,1.0,0.5000,0.0000,1.0
2,381,A 40-year-old man with neurofibromatosis type ...,C,0.043,C,correct,1.00,C,correct,1.00,...,<unused94>thought\nThe user wants me to act as...,<unused94>thought\nThe user wants me to analyz...,<unused94>thought\nThe user wants me to consol...,"```json\n{\n ""answer"": ""C"",\n ""confidence"": ...",The provided assessments strongly align. Both ...,0.6667,1.0,0.6667,0.3333,1.0


In [16]:
# ── Parse correctness ──
# Adapt column names to match your CSV
# Option A: If CSV has 'text_correct' and 'vision_correct' as 'correct'/'wrong'
if 'text_r1_correct' in df.columns:
    df['text_ok'] = df['text_r1_correct'].str.lower().str.strip() == 'correct'
    df['vision_ok'] = df['vision_r1_correct'].str.lower().str.strip() == 'correct'
    df['meta_ok'] = df['meta_correct'].str.lower().str.strip() == 'correct'

# Option B: If CSV has 'text_predicted' and 'ground_truth'
elif 'text_predicted' in df.columns and 'ground_truth' in df.columns:
    df['text_ok'] = df['text_predicted'].str.strip().str.upper() == df['ground_truth'].str.strip().str.upper()
    df['vision_ok'] = df['vision_predicted'].str.strip().str.upper() == df['ground_truth'].str.strip().str.upper()
    df['meta_ok'] = df['meta_predicted'].str.strip().str.upper() == df['ground_truth'].str.strip().str.upper()

# Option C: For LangGraph runs with 'text_initial_correct' / 'final_correct'
elif 'text_initial_correct' in df.columns:
    df['text_ok'] = df['text_initial_correct'].str.lower().str.strip() == 'correct'
    df['vision_ok'] = df['vision_correct'].str.lower().str.strip() == 'correct'
    df['meta_ok'] = df['final_correct'].str.lower().str.strip() == 'correct'

else:
    raise ValueError("Cannot find correctness columns. Check your CSV column names.")

print(f"Text correct: {df['text_ok'].sum()}/{len(df)} ({df['text_ok'].mean():.1%})")
print(f"Vision correct: {df['vision_ok'].sum()}/{len(df)} ({df['vision_ok'].mean():.1%})")
print(f"Meta correct: {df['meta_ok'].sum()}/{len(df)} ({df['meta_ok'].mean():.1%})")

Text correct: 374/689 (54.3%)
Vision correct: 222/689 (32.2%)
Meta correct: 322/689 (46.7%)


## 2. Compute Oracle Ceiling

For each case: if **either** the text agent OR the vision agent got the correct answer, the oracle selects correctly.

In [17]:
# Oracle: correct if EITHER specialist got it right
df['oracle_correct'] = df['text_ok'] | df['vision_ok']

# Also compute: both correct, only text, only vision, neither
df['both_correct'] = df['text_ok'] & df['vision_ok']
df['only_text'] = df['text_ok'] & ~df['vision_ok']
df['only_vision'] = ~df['text_ok'] & df['vision_ok']
df['neither'] = ~df['text_ok'] & ~df['vision_ok']

n = len(df)
oracle_count = df['oracle_correct'].sum()
oracle_pct = oracle_count / n * 100

print(f"\n{'='*50}")
print(f"ORACLE CEILING RESULTS ({n} cases)")
print(f"{'='*50}")
print(f"Oracle ceiling:    {oracle_count}/{n} = {oracle_pct:.1f}%")
print(f"  Both correct:    {df['both_correct'].sum()}/{n} = {df['both_correct'].mean():.1%}")
print(f"  Only text:       {df['only_text'].sum()}/{n} = {df['only_text'].mean():.1%}")
print(f"  Only vision:     {df['only_vision'].sum()}/{n} = {df['only_vision'].mean():.1%}")
print(f"  Neither correct: {df['neither'].sum()}/{n} = {df['neither'].mean():.1%}")
print(f"\nText alone:     {df['text_ok'].sum()}/{n} = {df['text_ok'].mean():.1%}")
print(f"Vision alone:   {df['vision_ok'].sum()}/{n} = {df['vision_ok'].mean():.1%}")
print(f"Meta (system):  {df['meta_ok'].sum()}/{n} = {df['meta_ok'].mean():.1%}")
print(f"\nArbitration gap: {oracle_pct:.1f}% - {df['meta_ok'].mean()*100:.1f}% = {oracle_pct - df['meta_ok'].mean()*100:.1f}pp")


ORACLE CEILING RESULTS (689 cases)
Oracle ceiling:    451/689 = 65.5%
  Both correct:    145/689 = 21.0%
  Only text:       229/689 = 33.2%
  Only vision:     77/689 = 11.2%
  Neither correct: 238/689 = 34.5%

Text alone:     374/689 = 54.3%
Vision alone:   222/689 = 32.2%
Meta (system):  322/689 = 46.7%

Arbitration gap: 65.5% - 46.7% = 18.7pp


## 3. Visualisation — Oracle Ceiling Breakdown

This figure shows where the diagnostic knowledge lives: how many cases each agent gets right, how many the oracle recovers, and how many are lost at the arbitration step.

In [18]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Left panel: Stacked breakdown of case outcomes ──
ax1 = axes[0]

categories = ['Both\ncorrect', 'Only text\ncorrect', 'Only vision\ncorrect', 'Neither\ncorrect']
counts = [
    df['both_correct'].sum(),
    df['only_text'].sum(),
    df['only_vision'].sum(),
    df['neither'].sum()
]
colors = ['#27AE60', '#5B9BD5', '#F39C12', '#E74C3C']

bars = ax1.bar(categories, counts, color=colors, edgecolor='white', linewidth=0.8, width=0.6)

for bar, count in zip(bars, counts):
    pct = count / n * 100
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{count}\n({pct:.1f}%)', ha='center', va='bottom',
             fontsize=11, fontweight='bold')

# Bracket for oracle ceiling
oracle_bar_height = max(counts[:3]) + 35
ax1.annotate('', xy=(0, oracle_bar_height), xytext=(2, oracle_bar_height),
            arrowprops=dict(arrowstyle='<->', color='#27AE60', lw=2))
ax1.text(1, oracle_bar_height + 8,
         f'Oracle ceiling: {oracle_count} cases ({oracle_pct:.1f}%)',
         ha='center', fontsize=11, fontweight='bold', color='#27AE60')

ax1.set_ylabel('Number of cases', fontsize=12, fontweight='bold')
ax1.set_title(f'Per-Case Diagnostic Knowledge ({n} cases)', fontsize=13, fontweight='bold')
ax1.set_ylim(0, max(counts) + 60)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── Right panel: Accuracy comparison with oracle ──
ax2 = axes[1]

labels = ['Vision\nagent', 'Meta\n(system)', 'Text\nagent', 'Oracle\nceiling']
accs = [
    df['vision_ok'].mean() * 100,
    df['meta_ok'].mean() * 100,
    df['text_ok'].mean() * 100,
    oracle_pct
]
bar_colors = ['#E74C3C', '#9B59B6', '#5B9BD5', '#27AE60']

bars2 = ax2.bar(labels, accs, color=bar_colors, edgecolor='white', linewidth=0.8, width=0.55)

for bar, acc in zip(bars2, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc:.1f}%', ha='center', va='bottom',
             fontsize=12, fontweight='bold')

# Arbitration gap annotation
meta_acc = df['meta_ok'].mean() * 100
gap = oracle_pct - meta_acc
mid_y = (meta_acc + oracle_pct) / 2
ax2.annotate('', xy=(3.3, oracle_pct), xytext=(3.3, meta_acc),
            arrowprops=dict(arrowstyle='<->', color='#333333', lw=1.5))
ax2.text(3.55, mid_y, f'{gap:.1f}pp\narbitration\ngap',
         ha='left', va='center', fontsize=10, fontweight='bold', color='#333333')

ax2.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax2.set_title('Accuracy vs Oracle Ceiling', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 80)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.yaxis.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('oracle_ceiling.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved to /workspace/figures/oracle_ceiling.png')

Saved to /workspace/figures/oracle_ceiling.png


C:\Users\Samruddha Dinde\AppData\Local\Temp\ipykernel_30668\2705144259.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Per-Case Heatmap (Optional)

Shows every case as a row: was text right? was vision right? did the oracle get it? did the system get it?
This makes the arbitration loss visible case-by-case.

In [ ]:
# Sort by difficulty: oracle-correct-but-system-wrong at top (the arbitration failures)
df['arbitration_failure'] = df['oracle_correct'] & ~df['meta_ok']
df_sorted = df.sort_values(['oracle_correct', 'meta_ok', 'text_ok'], ascending=[False, True, False])

# Build matrix for heatmap
matrix = df_sorted[['text_ok', 'vision_ok', 'oracle_correct', 'meta_ok']].values.astype(int)

fig, ax = plt.subplots(figsize=(6, 10))

# Custom colormap: red=wrong, green=correct
from matplotlib.colors import ListedColormap
cmap = ListedColormap(['#FADBD8', '#ABEBC6'])

im = ax.imshow(matrix, aspect='auto', cmap=cmap, interpolation='nearest')

ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Text\nagent', 'Vision\nagent', 'Oracle\n(either)', 'System\n(meta)'], fontsize=10)
ax.set_ylabel(f'Cases (sorted by oracle success, n={len(df)})', fontsize=11)
ax.set_title('Per-Case Correctness: Where Arbitration Loses Knowledge', fontsize=12, fontweight='bold')

# Annotate counts
arb_fail_count = df['arbitration_failure'].sum()
ax.text(3.7, len(df)*0.15, f'{arb_fail_count} cases\nwhere oracle\nis correct but\nsystem is wrong',
        fontsize=9, color='#C0392B', fontweight='bold', va='center')

plt.tight_layout()
plt.savefig('/workspace/figures/oracle_ceiling_heatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved to /workspace/figures/oracle_ceiling_heatmap.png')

In [ ]:
# Summary for thesis
print("\nSUMMARY FOR THESIS:")
print(f"Oracle ceiling: {oracle_count}/{n} ({oracle_pct:.1f}%)")
print(f"System accuracy: {df['meta_ok'].sum()}/{n} ({df['meta_ok'].mean()*100:.1f}%)")
print(f"Arbitration gap: {oracle_pct - df['meta_ok'].mean()*100:.1f} percentage points")
print(f"Arbitration failures (oracle correct, system wrong): {arb_fail_count}")
print(f"\nOf the {oracle_count} oracle-correct cases:")
print(f"  System recovered: {(df['oracle_correct'] & df['meta_ok']).sum()}")
print(f"  System lost:      {arb_fail_count}")
print(f"  Recovery rate:    {(df['oracle_correct'] & df['meta_ok']).sum()/oracle_count:.1%}")